Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Clone repo

In [4]:
import os
from google.colab import userdata

os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
token = os.environ['GITHUB_TOKEN']

!git clone https://{token}@github.com/DevDharmik/Smart-Building-Anomaly-Detection.git \
  /content/smart-building-anomaly-detection

!git -C /content/smart-building-anomaly-detection config user.email "dharmikchampaneri@gmail.com"
!git -C /content/smart-building-anomaly-detection config user.name "DevDharmik"
print("✅ Repo cloned")

Cloning into '/content/smart-building-anomaly-detection'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 55 (delta 18), reused 33 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 1.31 MiB | 5.14 MiB/s, done.
Resolving deltas: 100% (18/18), done.
✅ Repo cloned


Install

In [5]:
!pip install groq streamlit pandas numpy matplotlib seaborn -q
print("✅ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.3 MB/s eta 0:00:00
✅ Done


Imports + Groq setup


In [6]:
import os
import sqlite3
import pandas as pd
import numpy as np
from groq import Groq
from google.colab import userdata

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
client = Groq(api_key=os.environ['GROQ_API_KEY'])

print("✅ Groq client ready")

✅ Groq client ready


Load anomaly results

In [7]:
conn = sqlite3.connect('/content/drive/MyDrive/smart_building.db')

# Check what tables actually exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in DB:", tables['name'].tolist())

# Load from energy_features (Sprint 2) instead
df = pd.read_sql('SELECT * FROM energy_features', conn)
conn.close()

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['building_id', 'timestamp']).reset_index(drop=True)

# Recreate z_score column
df['z_score'] = (df['meter_reading'] - df['rolling_mean_24h']) / df['rolling_std_24h'].replace(0, 1)

print(f"✅ Loaded: {df.shape}")
df.head()

Tables in DB: ['energy_clean', 'energy_features', 'anomaly_explanations']
✅ Loaded: (4911000, 23)


,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,hour,...,day_of_year,rolling_mean_24h,rolling_std_24h,lag_7day,z_score,zscore_anomaly,iqr_lower,iqr_upper,iqr_anomaly,anomaly_label
0,0,0,2016-01-01 00:00:00,0.0,0,Education,7432,2008.0,NaN,0,...,1,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
1,0,0,2016-01-01 01:00:00,0.0,0,Education,7432,2008.0,NaN,1,...,1,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
2,0,0,2016-01-01 02:00:00,0.0,0,Education,7432,2008.0,NaN,2,...,1,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
3,0,0,2016-01-01 03:00:00,0.0,0,Education,7432,2008.0,NaN,3,...,1,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0
4,0,0,2016-01-01 04:00:00,0.0,0,Education,7432,2008.0,NaN,4,...,1,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0


Build explanation function

In [9]:
def explain_anomaly(row):
    def safe(val, fmt='.2f'):
        try:
            return format(float(val), fmt) if pd.notna(val) else 'N/A'
        except:
            return 'N/A'

    prompt = f"""
You are an energy analyst for a smart building management system.

An anomaly was detected with the following data:
- Building ID      : {row['building_id']}
- Timestamp        : {row['timestamp']}
- Meter Reading    : {safe(row['meter_reading'])} kWh
- 24hr Rolling Mean: {safe(row.get('rolling_mean_24h'))} kWh
- 24hr Rolling Std : {safe(row.get('rolling_std_24h'))} kWh
- Z-score          : {safe(row.get('z_score'))}
- 7-day Lag Value  : {safe(row.get('lag_7day'))} kWh
- Hour of Day      : {safe(row.get('hour'), '.0f')}
- Day of Week      : {safe(row.get('day_of_week'), '.0f')} (0=Mon, 6=Sun)
- Is Weekend       : {'Yes' if row.get('is_weekend') == 1 else 'No'}
- Isolation Forest : {'Yes' if row.get('if_anomaly') == 1 else 'No'}
- Z-score Flag     : {'Yes' if row.get('zscore_anomaly') == 1 else 'No'}
- IQR Flag         : {'Yes' if row.get('iqr_anomaly') == 1 else 'No'}

In 3-4 sentences explain: why this reading is anomalous, what the likely cause is, and what action a building manager should take.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=200
    )
    return response.choices[0].message.content.strip()

print("✅ explain_anomaly() ready — model: llama-3.3-70b-versatile")

✅ explain_anomaly() ready — model: llama-3.3-70b-versatile


Test on 3 anomalies

In [10]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

FEATURES = [
    'meter_reading', 'rolling_mean_24h', 'rolling_std_24h',
    'lag_7day', 'hour', 'day_of_week', 'month', 'is_weekend'
]

# Drop NaN + keep required columns
available_cols = [c for c in ['zscore_anomaly', 'iqr_anomaly', 'anomaly_label']
                  if c in df.columns]

df_model = df[FEATURES + ['building_id', 'timestamp'] + available_cols].dropna().copy()

# Scale
scaler   = MinMaxScaler()
X_scaled = scaler.fit_transform(df_model[FEATURES])

# Isolation Forest
iso_forest = IsolationForest(n_estimators=200, contamination=0.09,
                              random_state=42, n_jobs=-1)
iso_forest.fit(X_scaled)

df_model['if_anomaly'] = (iso_forest.predict(X_scaled) == -1).astype(int)
df_model['if_score']   = iso_forest.decision_function(X_scaled)
df_model['z_score']    = (df_model['meter_reading'] - df_model['rolling_mean_24h']) / \
                          df_model['rolling_std_24h'].replace(0, 1)

# Add zscore/iqr flags if missing
if 'zscore_anomaly' not in df_model.columns:
    df_model['zscore_anomaly'] = (df_model['z_score'].abs() > 3).astype(int)
if 'iqr_anomaly' not in df_model.columns:
    Q1 = df_model['meter_reading'].quantile(0.25)
    Q3 = df_model['meter_reading'].quantile(0.75)
    IQR = Q3 - Q1
    df_model['iqr_anomaly'] = (
        (df_model['meter_reading'] < Q1 - 1.5*IQR) |
        (df_model['meter_reading'] > Q3 + 1.5*IQR)
    ).astype(int)

df = df_model.copy()

print(f"✅ Done | shape: {df.shape}")
print(f"IF anomaly rate    : {df['if_anomaly'].mean()*100:.2f}%")
print(f"Z-score anomaly rate: {df['zscore_anomaly'].mean()*100:.2f}%")
print(f"IQR anomaly rate   : {df['iqr_anomaly'].mean()*100:.2f}%")
print(f"Columns: {df.columns.tolist()}")

✅ Done | shape: (4911000, 16)
IF anomaly rate    : 9.00%
Z-score anomaly rate: 0.79%
IQR anomaly rate   : 4.01%
Columns: ['meter_reading', 'rolling_mean_24h', 'rolling_std_24h', 'lag_7day', 'hour', 'day_of_week', 'month', 'is_weekend', 'building_id', 'timestamp', 'zscore_anomaly', 'iqr_anomaly', 'anomaly_label', 'if_anomaly', 'if_score', 'z_score']


Generate explanations for top 20 anomalies + save

In [11]:
import time

top_anomalies = df[
    (df['if_anomaly'] == 1) &
    (df['zscore_anomaly'] == 1)
].head(20).copy()

explanations = []
for i, (_, row) in enumerate(top_anomalies.iterrows()):
    print(f"Explaining anomaly {i+1}/20...", end=' ')
    try:
        exp = explain_anomaly(row)
        explanations.append(exp)
        print("✅")
    except Exception as e:
        explanations.append(f"Error: {e}")
        print(f"❌ → {e}")   # <-- show actual error
    time.sleep(2)            # <-- avoid rate limit (30 req/min)

top_anomalies['llm_explanation'] = explanations

conn = sqlite3.connect('/content/drive/MyDrive/smart_building.db')
top_anomalies.to_sql('anomaly_explanations', conn, if_exists='replace', index=False)
conn.close()

print(f"\n✅ Saved {len(top_anomalies)} explanations to SQLite")
top_anomalies[['building_id','timestamp','meter_reading','llm_explanation']].head()

Explaining anomaly 1/20... ✅
Explaining anomaly 2/20... ✅
Explaining anomaly 3/20... ✅
Explaining anomaly 4/20... ✅
Explaining anomaly 5/20... ✅
Explaining anomaly 6/20... ✅
Explaining anomaly 7/20... ✅
Explaining anomaly 8/20... ✅
Explaining anomaly 9/20... ✅
Explaining anomaly 10/20... ✅
Explaining anomaly 11/20... ✅
Explaining anomaly 12/20... ✅
Explaining anomaly 13/20... ✅
Explaining anomaly 14/20... ✅
Explaining anomaly 15/20... ✅
Explaining anomaly 16/20... ✅
Explaining anomaly 17/20... ✅
Explaining anomaly 18/20... ✅
Explaining anomaly 19/20... ✅
Explaining anomaly 20/20... ✅

✅ Saved 20 explanations to SQLite


,building_id,timestamp,meter_reading,llm_explanation
2923,0,2016-05-01 19:00:00,448.000,This meter reading is anomalous due to its sig...
29275,3,2016-05-01 19:00:00,937.000,This meter reading is anomalous due to its sig...
29916,3,2016-05-28 12:00:00,375.067,This meter reading is anomalous due to its sig...
31121,3,2016-07-17 17:00:00,315.343,This meter reading is anomalous due to its sig...
31628,3,2016-08-07 20:00:00,184.974,This meter reading is anomalous due to its sig...
